# SLR-RAG : Step 1

## Package import

In [1]:
# Import packages
%load_ext autoreload
%autoreload 2
import pandas as pd
from slrrag.querier_scopus import query_scopus

pd.__version__

'3.0.3'

## Inputs

### Query

In [2]:
main_query = """
("risk assessment" OR "risk analysis") AND
("Offshore windfarm" OR "Offshore windmill" OR "Offshore wind turbine" OR "Offshore wind park" OR "OWF") AND
("Strategic" OR "Static" OR "Long-term" OR "Marine spatial planning" OR "Large scale") AND
("maritime" OR  "navigational" OR "nautical")
"""

### Reference papers
Represent them as (TITLE,DOI)
The notebook will try to do the merge based on the DOI first, and on the title otherwise if the paper has no DOI.

In [3]:
from slrrag.settings import SchemaPapers


reference_papers = [
    (
        "A dynamic risk assessment method to address safety of navigation concerns around offshore renewable energy installations",
        "10.1177/1475090219837409",
    ),
    (
        "A Marine GIS-based Alert System to Prevent Vessels Collision with Offshore Platforms",
        "10.51400/2709-6998.2462",
    ),
]
df_reference_paper = pd.DataFrame(
    reference_papers,
    columns=[
        "title",
        "doi",
    ],
)
df_reference_paper = SchemaPapers.validate(df_reference_paper, lazy=True)
df_reference_paper

,title,doi,type,year,date,abstract
0,A dynamic risk assessment method to address sa...,10.1177/1475090219837409,NaN,<NA>,NaT,NaN
1,A Marine GIS-based Alert System to Prevent Ves...,10.51400/2709-6998.2462,NaN,<NA>,NaT,NaN


## Query databases

### Add scopus papers

In [4]:
df_scopus = query_scopus(main_query)
df_scopus

Query : 
TITLE-ABS-KEY( 
("risk assessment" OR "risk analysis") AND
("Offshore windfarm" OR "Offshore windmill" OR "Offshore wind turbine" OR "Offshore wind park" OR "OWF") AND
("Strategic" OR "Static" OR "Long-term" OR "Marine spatial planning" OR "Large scale") AND
("maritime" OR  "navigational" OR "nautical")
)

3 found in Scopus.
TITLE-ABS-KEY( 
("risk assessment" OR "risk analysis") AND
("Offshore windfarm" OR "Offshore windmill" OR "Offshore wind turbine" OR "Offshore wind park" OR "OWF") AND
("Strategic" OR "Static" OR "Long-term" OR "Marine spatial planning" OR "Large scale") AND
("maritime" OR  "navigational" OR "nautical")
)
":


100%|██████████| 1/1 [00:00<?, ?it/s]


,doi,title,date,abstract,id_scopus,type,year,keywords_scopus,relevance_order_scopus
0,10.1016/j.apor.2025.104564,Multi-perspective nautical safety risk assessm...,2025-05-01,"The increasing amount of activities at sea, in...",2-s2.0-105003177564,article,<NA>,Allision risks | Event table | Mitigation meas...,0
1,10.3390/su17188252,Evaluation of Ship Importance in Offshore Wind...,2025-09-01,With the rapid expansion of offshore wind farm...,2-s2.0-105017137650,article,<NA>,complex network | gravity model | offshore win...,1
2,NaN,Determining collision risks for fixed offshore...,2020-01-01,To determine the risk involved in operations a...,2-s2.0-85096516091,conference-paper,<NA>,AIS | Drifting collision frequencies | Maritim...,2


In [5]:
from slrrag.main import merge_papers

df_papers = merge_papers(df_reference_paper,df_scopus,"scopus")
df_papers

,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,relevance_order_scopus
0,Multi-perspective nautical safety risk assessm...,10.1016/j.apor.2025.104564,article,<NA>,2025-05-01,"The increasing amount of activities at sea, in...",2-s2.0-105003177564,Allision risks | Event table | Mitigation meas...,0
1,A dynamic risk assessment method to address sa...,10.1177/1475090219837409,NaN,<NA>,NaT,NaN,NaN,NaN,<NA>
2,Evaluation of Ship Importance in Offshore Wind...,10.3390/su17188252,article,<NA>,2025-09-01,With the rapid expansion of offshore wind farm...,2-s2.0-105017137650,complex network | gravity model | offshore win...,1
3,A Marine GIS-based Alert System to Prevent Ves...,10.51400/2709-6998.2462,NaN,<NA>,NaT,NaN,NaN,NaN,<NA>
4,Determining collision risks for fixed offshore...,NaN,conference-paper,<NA>,2020-01-01,To determine the risk involved in operations a...,2-s2.0-85096516091,AIS | Drifting collision frequencies | Maritim...,2


### Add OpenAlex papers

In [6]:
from slrrag.querier_openalex import query_openalex
df_openalex = query_openalex(main_query)
df_openalex

Query : 
("risk assessment" OR "risk analysis") AND
("Offshore windfarm" OR "Offshore windmill" OR "Offshore wind turbine" OR "Offshore wind park" OR "OWF") AND
("Strategic" OR "Static" OR "Long-term" OR "Marine spatial planning" OR "Large scale") AND
("maritime" OR  "navigational" OR "nautical")

5 found in Scopus.


1it [00:00,  1.13it/s]                 


,doi,id_openalex,title,date,abstract,type,year,relevance_openalex,pdf,topic_openalex,fields_openalex,subfields_openalex,domain_openalex,keywords_openalex
0,10.1016/j.apor.2025.104564,https://openalex.org/W4409739111,Multi-perspective nautical safety risk assessm...,2025-04-24,"The increasing amount of activities at sea, in...",article,<NA>,102.223070,https://content.openalex.org/works/W4409739111...,Maritime Navigation and Safety,Engineering,Ocean Engineering,Physical Sciences,Offshore wind power|Submarine pipeline|Perspec...
1,NaN,https://openalex.org/W638864640,Maritime engineering and technology : proceedi...,2012-01-01,Ship traffic Maritime situational awareness in...,book,<NA>,20.300217,https://content.openalex.org/works/W638864640.pdf,Maritime Transport Emissions and Efficiency,Environmental Science,Environmental Engineering,Physical Sciences,Engineering|Political science|Engineering mana...
2,10.1115/1.4053700,https://openalex.org/W4210662789,Profiles of Two JOMAE Associate Editors (The F...,2022-01-28,"For 35 years, beginning in 1987, the ASME Jour...",article,<NA>,2.161095,https://content.openalex.org/works/W4210662789...,Offshore Engineering and Technologies,Engineering,Ocean Engineering,Physical Sciences,China|Associate editor|Library science|Politic...
3,10.3390/su17188252,https://openalex.org/W4414206190,Evaluation of Ship Importance in Offshore Wind...,2025-09-14,With the rapid expansion of offshore wind farm...,article,<NA>,1.998297,https://content.openalex.org/works/W4414206190...,Maritime Navigation and Safety,Engineering,Ocean Engineering,Physical Sciences,Offshore wind power|Submarine pipeline|Complex...
4,10.1002/qre.3611,https://openalex.org/W4400159364,Advanced reliability and safety methodologies ...,2024-06-28,Modern engineering systems tend to be more com...,article,<NA>,1.625039,https://content.openalex.org/works/W4400159364...,Risk and Safety Analysis,Decision Sciences,"Statistics, Probability and Uncertainty",Social Sciences,Reliability engineering|Reliability (semicondu...


In [7]:
df_papers = merge_papers(df_papers,df_openalex,"openalex")
df_papers

,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,relevance_order_scopus,id_openalex,relevance_openalex,pdf,topic_openalex,fields_openalex,subfields_openalex,domain_openalex,keywords_openalex
0,Advanced reliability and safety methodologies ...,10.1002/qre.3611,article,<NA>,2024-06-28,Modern engineering systems tend to be more com...,NaN,NaN,<NA>,https://openalex.org/W4400159364,1.625039,https://content.openalex.org/works/W4400159364...,Risk and Safety Analysis,Decision Sciences,"Statistics, Probability and Uncertainty",Social Sciences,Reliability engineering|Reliability (semicondu...
1,Multi-perspective nautical safety risk assessm...,10.1016/j.apor.2025.104564,article,<NA>,2025-05-01,"The increasing amount of activities at sea, in...",2-s2.0-105003177564,Allision risks | Event table | Mitigation meas...,0,https://openalex.org/W4409739111,102.223070,https://content.openalex.org/works/W4409739111...,Maritime Navigation and Safety,Engineering,Ocean Engineering,Physical Sciences,Offshore wind power|Submarine pipeline|Perspec...
2,Profiles of Two JOMAE Associate Editors (The F...,10.1115/1.4053700,article,<NA>,2022-01-28,"For 35 years, beginning in 1987, the ASME Jour...",NaN,NaN,<NA>,https://openalex.org/W4210662789,2.161095,https://content.openalex.org/works/W4210662789...,Offshore Engineering and Technologies,Engineering,Ocean Engineering,Physical Sciences,China|Associate editor|Library science|Politic...
3,A dynamic risk assessment method to address sa...,10.1177/1475090219837409,NaN,<NA>,NaT,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Evaluation of Ship Importance in Offshore Wind...,10.3390/su17188252,article,<NA>,2025-09-01,With the rapid expansion of offshore wind farm...,2-s2.0-105017137650,complex network | gravity model | offshore win...,1,https://openalex.org/W4414206190,1.998297,https://content.openalex.org/works/W4414206190...,Maritime Navigation and Safety,Engineering,Ocean Engineering,Physical Sciences,Offshore wind power|Submarine pipeline|Complex...
5,A Marine GIS-based Alert System to Prevent Ves...,10.51400/2709-6998.2462,NaN,<NA>,NaT,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Determining collision risks for fixed offshore...,NaN,conference-paper,<NA>,2020-01-01,To determine the risk involved in operations a...,2-s2.0-85096516091,AIS | Drifting collision frequencies | Maritim...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Maritime engineering and technology : proceedi...,NaN,book,<NA>,2012-01-01,Ship traffic Maritime situational awareness in...,NaN,NaN,<NA>,https://openalex.org/W638864640,20.300217,https://content.openalex.org/works/W638864640.pdf,Maritime Transport Emissions and Efficiency,Environmental Science,Environmental Engineering,Physical Sciences,Engineering|Political science|Engineering mana...


### Add Lens

In [11]:
from slrrag.main import print_lens_export_info
from slrrag.query_converter import convert_query_to_lens
from slrrag.settings import PATH_LENS_EXPORT

lens_export_file = PATH_LENS_EXPORT
lens_query = convert_query_to_lens(main_query)
print_lens_export_info(lens_query,lens_export_file)


Lens query:

((title:("risk assessment") OR abstract:("risk assessment")) OR (title:("risk analysis") OR abstract:("risk analysis"))) AND
((title:("Offshore windfarm") OR abstract:("Offshore windfarm")) OR (title:("Offshore windmill") OR abstract:("Offshore windmill")) OR (title:("Offshore wind turbine") OR abstract:("Offshore wind turbine")) OR (title:("Offshore wind park") OR abstract:("Offshore wind park")) OR (title:("OWF") OR abstract:("OWF"))) AND
((title:("Strategic") OR abstract:("Strategic")) OR (title:("Static") OR abstract:("Static")) OR (title:("Long-term") OR abstract:("Long-term")) OR (title:("Marine spatial planning") OR abstract:("Marine spatial planning")) OR (title:("Large scale") OR abstract:("Large scale"))) AND
((title:("maritime") OR abstract:("maritime")) OR  (title:("navigational") OR abstract:("navigational")) OR (title:("nautical") OR abstract:("nautical")))


Please copy the query above and paste it into the query editor at:
https://www.lens.org/lens/search/

In [12]:
from slrrag.querier_lens import read_lens_export
df_lens = read_lens_export(lens_export_file)
df_lens

,doi,title,date,abstract,type,year,id_lens,topic_lens,keywords_lens
0,10.1016/j.apor.2025.104564,Multi-perspective nautical safety risk assessm...,NaT,"The increasing amount of activities at sea, in...",article,<NA>,017-608-973-984-054,Offshore wind power; Submarine pipeline; Persp...,NaN
1,NaN,Maritime engineering and technology : proceedi...,NaT,Ship traffic Maritime situational awareness in...,book,<NA>,080-703-447-465-297,Engineering; Marine engineering; Marine techno...,NaN
2,10.1115/1.4053700,Profiles of Two JOMAE Associate Editors (The F...,2022-02-18,"For 35 years, beginning in 1987, the ASME Jour...",article,<NA>,174-185-838-473-498,China; Associate editor; Library science; Poli...,NaN
3,10.1002/qre.3611,Advanced reliability and safety methodologies ...,2024-06-28,Modern engineering systems tend to be more com...,article,<NA>,187-502-147-124-430,Reliability engineering; Reliability (semicond...,NaN
4,10.3390/su17188252,Evaluation of Ship Importance in Offshore Wind...,2025-09-14,<jats:p>With the rapid expansion of offshore w...,article,<NA>,196-025-595-838-127,Offshore wind power; Submarine pipeline; Marin...,NaN


In [13]:
df_papers = merge_papers(df_papers,df_lens,"lens")
df_papers

,title,doi,type,year,date,abstract,id_scopus,keywords_scopus,relevance_order_scopus,id_openalex,relevance_openalex,pdf,topic_openalex,fields_openalex,subfields_openalex,domain_openalex,keywords_openalex,id_lens,topic_lens,keywords_lens
0,Advanced reliability and safety methodologies ...,10.1002/qre.3611,article,<NA>,2024-06-28,Modern engineering systems tend to be more com...,NaN,NaN,<NA>,https://openalex.org/W4400159364,1.625039,https://content.openalex.org/works/W4400159364...,Risk and Safety Analysis,Decision Sciences,"Statistics, Probability and Uncertainty",Social Sciences,Reliability engineering|Reliability (semicondu...,187-502-147-124-430,Reliability engineering; Reliability (semicond...,NaN
1,Multi-perspective nautical safety risk assessm...,10.1016/j.apor.2025.104564,article,<NA>,2025-05-01,"The increasing amount of activities at sea, in...",2-s2.0-105003177564,Allision risks | Event table | Mitigation meas...,0,https://openalex.org/W4409739111,102.223070,https://content.openalex.org/works/W4409739111...,Maritime Navigation and Safety,Engineering,Ocean Engineering,Physical Sciences,Offshore wind power|Submarine pipeline|Perspec...,017-608-973-984-054,Offshore wind power; Submarine pipeline; Persp...,NaN
2,Profiles of Two JOMAE Associate Editors (The F...,10.1115/1.4053700,article,<NA>,2022-01-28,"For 35 years, beginning in 1987, the ASME Jour...",NaN,NaN,<NA>,https://openalex.org/W4210662789,2.161095,https://content.openalex.org/works/W4210662789...,Offshore Engineering and Technologies,Engineering,Ocean Engineering,Physical Sciences,China|Associate editor|Library science|Politic...,174-185-838-473-498,China; Associate editor; Library science; Poli...,NaN
3,A dynamic risk assessment method to address sa...,10.1177/1475090219837409,NaN,<NA>,NaT,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Evaluation of Ship Importance in Offshore Wind...,10.3390/su17188252,article,<NA>,2025-09-01,With the rapid expansion of offshore wind farm...,2-s2.0-105017137650,complex network | gravity model | offshore win...,1,https://openalex.org/W4414206190,1.998297,https://content.openalex.org/works/W4414206190...,Maritime Navigation and Safety,Engineering,Ocean Engineering,Physical Sciences,Offshore wind power|Submarine pipeline|Complex...,196-025-595-838-127,Offshore wind power; Submarine pipeline; Marin...,NaN
5,A Marine GIS-based Alert System to Prevent Ves...,10.51400/2709-6998.2462,NaN,<NA>,NaT,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Determining collision risks for fixed offshore...,NaN,conference-paper,<NA>,2020-01-01,To determine the risk involved in operations a...,2-s2.0-85096516091,AIS | Drifting collision frequencies | Maritim...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Maritime engineering and technology : proceedi...,NaN,book,<NA>,2012-01-01,Ship traffic Maritime situational awareness in...,NaN,NaN,<NA>,https://openalex.org/W638864640,20.300217,https://content.openalex.org/works/W638864640.pdf,Maritime Transport Emissions and Efficiency,Environmental Science,Environmental Engineering,Physical Sciences,Engineering|Political science|Engineering mana...,080-703-447-465-297,Engineering; Marine engineering; Marine techno...,NaN


### Add dimensions